In [4]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [5]:
import spectral_analysis as spec
import usv_library as ul
import setup
import os
import pandas as pd
import numpy as np

In [2]:
# Set path variables:
repo_path = os.getcwd()
USV_DATA_path = os.path.join(repo_path, 'USV_DATA')
wavfiles_path = os.path.join(repo_path, 'wavfiles')

### Make list of files that both have a local wavfile and a rated file

In [87]:
# View session folders in wavfiles directory for easy access
pd.set_option('display.max_colwidth', None)
pd.DataFrame({"session_folder":
              [subdir for subdir in os.listdir(wavfiles_path) if os.path.isdir(os.path.join(wavfiles_path, subdir))]}).sort_values('session_folder')

,session_folder
9,SK2aSK1b_Restrainer_Experiment_Other_d5_s1_12022025
4,SK2aSK2b_Restrainer_Experiment_Cagemate2_d9_s1_12062025
13,SK2aSK2b_Restrainer_Experiment_Cagemate_d1_s1_11252025
2,SK2aSK2b_Restrainer_Experiment_Divided_d4_s1_11282025
17,SK3aSK1b_Restrainer_Experiment_Other _d6_s1_06272025_nontether
3,SK3aSK3b_Restrainer_Experiment_Cagemate2 _d9_s1_07042025
15,SK3aSK3b_Restrainer_Experiment_Cagemate_d1_s2_06202025
12,SK3aSK3b_Restrainer_Experiment_Divided_d3_s1_06242025
11,SK3bSK3a_Restrainer_Experiment_Divided_d6_s1_05142025_nontether
14,SK3bSK3a_Restrainer_Experiment_Divided_d6_s2_05142025


## NExt

In [113]:
# Session ID entry
session_id = setup.get_input("Session ID", "Enter the session name: ")
print(f"Session ID entered: {session_id} \n")

# Look for WAV file -- both detection and verification need it
wavfile = os.path.join(wavfiles_path, session_id, "Dodotronic Track.wav")
if not os.path.exists(wavfile):
    raise FileNotFoundError(f"WAV file not found, cannot continue. Missing: {wavfile}")
else:
    print(f"WAV file found: {wavfile} \n Looking for rated file...")
    rated_path = setup.find_file("rated", os.path.join(USV_DATA_path, session_id))
    if rated_path:
        if len(os.listdir(rated_path)) > 0:
            rater_file = True
            print(f"Rater files found:")
            for file in os.listdir(rated_path):
                rater_file_name = file
                print(f"- {file}")
        else:
            rater_file = False
            print("Rated folder found but it is empty, cannot continue.")
    else:
        rater_file = False
        print("Rated folder not found, cannot continue.")

Session ID entered: Sk5aSK5b_Restrainer_Experiment_Cagemate_d2_s1_041625 

WAV file found: /Users/sophieneale/Code/projects_work/usv_detection/wavfiles/Sk5aSK5b_Restrainer_Experiment_Cagemate_d2_s1_041625/Dodotronic Track.wav 
 Looking for rated file...
Rater files found:
- Sk5aSK5b_Restrainer_Experiment_Cagemate_d2_s1_041625_USV_SN.csv


In [114]:
# Choose rated file (if applicable)
if rater_file and len(os.listdir(rated_path)) > 1:
    rater_file_name = setup.get_input("Rater file name", "Multiple rater files found. Please enter the name of the rater file you want to use: ")
    
    # if not os.path.exists(rater_file_path):
    #     raise FileNotFoundError(f"Rater file not found, cannot continue. Missing: {rater_file_path}")
    # else:
    #     print(f"Rater file found: {rater_file_path}")

rater_file_path = os.path.join(rated_path, rater_file_name)

In [115]:
# Find CSV
csv_path = setup.find_file(f"{session_id}_USV.csv", os.path.join(USV_DATA_path, session_id))

In [116]:
if csv_path and rater_file_path:
    print(f"CSV file found: {csv_path}")
    print(f"Rater file found: {rater_file_path}")
    data = pd.read_csv(csv_path)
    rated = pd.read_csv(rater_file_path, header=0, names=['label'])
    data["25_kHz_call"] = data['label'].isin(rated['label'])


CSV file found: /Users/sophieneale/Code/projects_work/usv_detection/USV_DATA/Sk5aSK5b_Restrainer_Experiment_Cagemate_d2_s1_041625/25kHz/25kHz_with_trial_starts_V2/Sk5aSK5b_Restrainer_Experiment_Cagemate_d2_s1_041625_USV.csv
Rater file found: /Users/sophieneale/Code/projects_work/usv_detection/USV_DATA/Sk5aSK5b_Restrainer_Experiment_Cagemate_d2_s1_041625/25kHz/25kHz_with_trial_starts_V2/rated/Sk5aSK5b_Restrainer_Experiment_Cagemate_d2_s1_041625_USV_SN.csv


In [117]:
%%capture
spec.get_from_rated(wavfile, data, session_id)

# STOP


In [ ]:
twentyfive = []
for usv in data['label']:
    twentyfive.append(int(usv in rated.to_numpy().flatten()))

In [4]:
spec.pad_spectrograms("spectrograms/noise")